This notebook implements the second stage of the preprocessing workflow for the
SWOT RiverSP obstruction analysis. Its primary objective is to transform the
intermediate node-level datasets derived from the original **SWOT L2 HR RiverSP**
products into application-ready datasets suitable for downstream analysis,
model development, and public data release.

Starting from the daily node-level datasets generated by
`scripts/01_preprocess/01_nodePre.py`, this notebook performs a series of
processing steps to organize SWOT river observations into structured,
analysis-ready data products. These steps include continent-level extraction of
ice-free observations, quality filtering, temporal coverage screening,
dataset partitioning at the reach level, and the derivation of node-level
anxillary variables.

The final outputs include **train, valid** and **test** datasets prepared at
the reach level to prevent information leakage between subsets. These datasets
represent the application-ready products used in the study and will be
archived and publicly released through **Zenodo** to support transparency and
reproducibility.

In addition to the train/test datasets, this workflow derives several node-level
anxillary variables also availabel at **Zenodo** such as:

- mean radar backscatter coefficient (σ₀ / `rdr_sig0`)
- mean **Global Surface Water (GSW) occurrence** values

These variables provide spatial and environmental context for SWOT river nodes
and are used as supporting features in subsequent analysis.

Intermediate preprocessing datasets are not distributed with the repository
due to their large size, but they can be regenerated using the provided
workflow.

In [1]:
from pathlib import Path
from glob import glob
import dask

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely import wkt
import rioxarray

# 1 Data preprocessing: continent-level node datasets

This step extracts records without river ice from the daily 
node-level CSV files generated by
`scripts/01_preprocess/01_nodePre.py` and 
converts them into continent CSV files for further analysis.

**Input**  
Derived daily node data:  
data/derived_raw/RiverSP_VC_daily/  
&nbsp;&nbsp;&nbsp;&nbsp;node_YYYYMMDD.csv  

**Output**  
Continent-specific node datasets (ice-free observations):  
data/derived_raw/continent/  
&nbsp;&nbsp;&nbsp;&nbsp;node_af_noice.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_eu_noice.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_si_noice.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_as_noice.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_au_noice.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_sa_noice.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_na_noice.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_ar_noice.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_gr_noice.csv  

**Notes**  
The datasets generated in this step are intermediate preprocessing results.
Due to their large size, they are not distributed with the repository but can
be regenerated using the provided scripts.


In [13]:
# Project root directory
PROJECT_ROOT = Path().resolve().parents[1]

# Input directory containing daily node CSV files
INPUT_DIR = PROJECT_ROOT / "data" / "derived_raw" / "RiverSP_VC_daily"

# Output directory for continent-level files
OUTPUT_DIR = PROJECT_ROOT / "data" / "derived_raw" / "continent"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [14]:
PROJECT_ROOT

WindowsPath('D:/github/SWOT_RiverSP_Obstruction')

In [ ]:
# Date range to process
dates = pd.date_range(start="2023-07-26", end="2025-05-03")
date_strings = dates.strftime("%Y%m%d").tolist()

# Mapping from continent code to short name
continent_map = {
    1: "af",
    2: "eu",
    3: "si",
    4: "as",
    5: "au",
    6: "sa",
    7: "na",
    8: "ar",
    9: "gr",
}

# Columns to keep
columns_to_keep = [
    "node_id",
    "time",
    "wse",
    "wse_u",
    "width",
    "ice_clim_f",
    "dark_frac",
    "rdr_sig0",
    "node_q_b",
]

In [ ]:
# Store data for each continent separately
continent_dfs = {code: [] for code in continent_map.keys()}

for date in date_strings:
    file_path = INPUT_DIR / f"node_{date}.csv"

    if not file_path.exists():
        continue

    df = pd.read_csv(file_path)

    # Remove invalid WSE values
    df = df[df["wse"] > -1e5]

    if df.empty:
        continue

    # Derive reach_id from node_id
    df["reach_id"] = df["node_id"] // 10000 * 10 + df["node_id"] % 10

    # Derive continent code from reach_id
    df["continent"] = (df["reach_id"] // 1e10).astype(int)

    # Remove ice-covered observations
    df = df[df["ice_clim_f"] == 0]

    if df.empty:
        continue

    # Keep only required columns
    df = df[columns_to_keep + ["continent"]]

    # Append data to each continent-specific list
    for code in continent_map.keys():
        continent_df = df[df["continent"] == code]

        if not continent_df.empty:
            continent_dfs[code].append(continent_df[columns_to_keep])

    print(f"Processed {date}")

In [ ]:
for code, short_name in continent_map.items():
    if continent_dfs[code]:
        merged_df = pd.concat(continent_dfs[code], ignore_index=True)
        output_file = OUTPUT_DIR / f"node_{short_name}_noice.csv"
        merged_df.to_csv(output_file, index=False)
        print(f"Saved {output_file}")
    else:
        print(f"No valid data for continent {code} ({short_name})")

# 2 Initial filtering of continent-level node data

This step performs an initial quality filtering of the continent-level node
datasets generated in the previous step. It removes duplicated observations,
applies basic quality thresholds, and retains only valid river reaches for
subsequent analysis.

**Input**  
Continent-level node datasets (ice-free observations):  
data/derived_raw/continent/  
&nbsp;&nbsp;&nbsp;&nbsp;node_af_noice.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_eu_noice.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_si_noice.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_as_noice.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_au_noice.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_sa_noice.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_na_noice.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_ar_noice.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_gr_noice.csv  

**Output**  
Filtered continent-level node datasets:  
data/derived_raw/continent_filtered/  
&nbsp;&nbsp;&nbsp;&nbsp;node_af_v5.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_eu_v5.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_si_v5.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_as_v5.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_au_v5.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_sa_v5.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_na_v5.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_ar_v5.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_gr_v5.csv  

**Notes**  
This step performs an initial quality control of the RiverSP node observations.
The filtered datasets remain intermediate preprocessing products and are not
included in the repository due to their large size. They can be regenerated
using the preprocessing workflow provided in this project.

In [ ]:
# Continent short names
name_list = ["af", "eu", "si", "as", "au", "sa", "na", "ar", "gr"]

# Input/output directories
INPUT_DIR = PROJECT_ROOT / "data" / "derived_raw" / "continent"
OUTPUT_DIR = PROJECT_ROOT / "data" / "derived_raw" / "continent_filtered"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
for name in name_list:
    input_file = INPUT_DIR / f"node_{name}_noice.csv"
    output_file = OUTPUT_DIR / f"node_{name}_v5.csv"

    if not input_file.exists():
        print(f"File not found: {input_file}")
        continue

    # Read continent-level node data
    df = pd.read_csv(input_file)
    print(f"Processing {name}...")

    # Remove duplicated node-time observations
    df = df.drop_duplicates(subset=["node_id", "time"])

    # Derive reach_id from node_id
    df["reach_id"] = df["node_id"] // 10000 * 10 + df["node_id"] % 10

    # Apply basic quality filters
    df = df[
        (df["wse_u"] < 0.5) &
        (df["ice_clim_f"] == 0) &
        (df["dark_frac"] < 0.9)
    ]

    # Keep only reaches with at least 10 node observations
    valid_reach_ids = (
        df.groupby("reach_id")
        .filter(lambda x: len(x) >= 10)["reach_id"]
        .unique()
    )
    df = df[df["reach_id"].isin(valid_reach_ids)]

    # Keep only river reaches (reach_type == 1)
    df["reach_type"] = df["reach_id"] % 10
    df = df[df["reach_type"] == 1]

    # Save selected columns after initial filtering
    df[
        [
            "node_id",
            "time",
            "wse",
            "wse_u",
            "width",
            "ice_clim_f",
            "dark_frac",
            "rdr_sig0",
            "node_q_b",
        ]
    ].to_csv(output_file, index=False)

    print(f"Saved to {output_file}")

# 3 Filter reaches by valid observation dates

This step further filters the continent-level node datasets generated in the
previous step by retaining only reaches with sufficient valid observation dates.

For each `reach_id`, the number of unique participating nodes is first counted.
A date is considered valid if the number of observed nodes on that date exceeds
90% of the total unique nodes for that reach. Only reaches with at least four
valid dates are retained.

**Input**  
Filtered continent-level node datasets:  
data/derived_raw/continent_filtered/  
&nbsp;&nbsp;&nbsp;&nbsp;node_af_v5.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_eu_v5.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_si_v5.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_as_v5.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_au_v5.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_sa_v5.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_na_v5.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_ar_v5.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_gr_v5.csv  

**Output**  
Continent-level node datasets with valid observation dates:  
data/derived_raw/continent_valid_dates/  
&nbsp;&nbsp;&nbsp;&nbsp;node_af_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_eu_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_si_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_as_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_au_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_sa_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_na_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_ar_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_gr_v5.10.csv  

**Notes**  
This step removes reaches with insufficient temporal coverage or incomplete node
participation. The generated files remain intermediate preprocessing products
and are not distributed with the repository due to their large size.

In [ ]:
# Define a function to retain only reaches with sufficient valid dates
def filter_valid_dates(group):
    # Count the total number of unique nodes within this reach
    unique_nodes = group["node_id"].nunique()
    threshold = 0.9 * unique_nodes

    # Count the number of unique observed nodes for each date
    date_counts = group.groupby("date")["node_id"].nunique()

    # A date is considered valid if node participation exceeds 90% of total nodes
    valid_dates = date_counts[date_counts > threshold].index

    # Retain only reaches with at least four valid dates
    if len(valid_dates) < 4:
        return pd.DataFrame()

    return group[group["date"].isin(valid_dates)]

In [ ]:
# Continent short names
name_list = ["af", "eu", "si", "as", "au", "sa", "na", "ar", "gr"]

# Input/output directories
INPUT_DIR = PROJECT_ROOT / "data" / "derived_raw" / "continent_filtered"
OUTPUT_DIR = PROJECT_ROOT / "data" / "derived_raw" / "continent_valid_dates"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for name in name_list:
    input_file = INPUT_DIR / f"node_{name}_v5.csv"
    output_file = OUTPUT_DIR / f"node_{name}_v5.10.csv"

    if not input_file.exists():
        print(f"File not found: {input_file}")
        continue

    # Read filtered continent-level node data
    df = pd.read_csv(input_file)
    print(f"Processing {name}...")

    # Derive reach_id from node_id
    df["reach_id"] = df["node_id"] // 10000 * 10 + df["node_id"] % 10

    # Convert SWOT time to calendar date
    df["time_str"] = pd.to_datetime(df["time"], unit="s", origin="2000-01-01")
    df["date"] = df["time_str"].dt.floor("D")
    df["month"] = df["date"].dt.month

    # Apply valid-date filtering within each reach
    df = (
        df.groupby("reach_id", group_keys=False)
        .apply(filter_valid_dates)
        .reset_index(drop=True)
    )

    # Save selected columns
    df[
        [
            "node_id",
            "time",
            "wse",
            "width",
            "ice_clim_f",
            "dark_frac",
            "rdr_sig0",
            "node_q_b",
            "wse_u",
        ]
    ].to_csv(output_file, index=False)

    print(f"Saved to {output_file}")

# 4 Split the datasets into train_valid and test sets


This step divides the filtered continent-level node datasets into a
`train_valid` subset and a `test` subset based on `reach_id`.

A random 10% sample of unique reaches is selected and assigned to the
`train_valid` subset. All remaining reaches are assigned to the `test` subset.
The split is performed at the reach level to ensure that observations from the
same reach do not appear in both subsets.

**Input**  
Continent-level node datasets with valid observation dates:  
data/derived_raw/continent_valid_dates/  
&nbsp;&nbsp;&nbsp;&nbsp;node_af_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_eu_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_si_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_as_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_au_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_sa_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_na_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_ar_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_gr_v5.10.csv  

**Output**  
Combined train_valid dataset:  
data/input_data/node_train_valid_v5.10.csv  

Continent-specific test datasets:  
data/input_data/  
&nbsp;&nbsp;&nbsp;&nbsp;node_af_test_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_eu_test_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_si_test_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_as_test_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_au_test_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_sa_test_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_na_test_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_ar_test_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_gr_test_v5.10.csv  

**Notes**  
The split is performed at the reach level rather than the observation level to
avoid data leakage between the `train_valid` and `test` subsets.

In [ ]:
# Continent short names
name_list = ["af", "eu", "si", "as", "au", "sa", "na", "ar", "gr"]

# Input/output directories
INPUT_DIR = PROJECT_ROOT / "data" / "derived_raw" / "continent_valid_dates"
OUTPUT_DIR = PROJECT_ROOT / "data" / "input_data" 

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Collect all unique reach IDs across all continent datasets
all_reach_ids = []

for name in name_list:
    input_file = INPUT_DIR / f"node_{name}_v5.10.csv"

    if not input_file.exists():
        print(f"File not found: {input_file}")
        continue

    df = pd.read_csv(input_file)
    print(f"Reading {name}...")

    # Derive reach_id from node_id
    df["reach_id"] = df["node_id"] // 10000 * 10 + df["node_id"] % 10

    all_reach_ids.append(df[["reach_id"]].drop_duplicates())

all_reach_ids = pd.concat(all_reach_ids, ignore_index=True).drop_duplicates()

# Randomly sample 10% of unique reaches for the train_valid subset
sampled_ids = all_reach_ids["reach_id"].sample(frac=0.10, random_state=42)

print(f"Number of sampled reaches: {len(sampled_ids)}")

In [ ]:
# Build the combined train_valid dataset
train_valid_dfs = []

for name in name_list:
    input_file = INPUT_DIR / f"node_{name}_v5.10.csv"

    if not input_file.exists():
        continue

    df = pd.read_csv(input_file)

    # Derive reach_id from node_id
    df["reach_id"] = df["node_id"] // 10000 * 10 + df["node_id"] % 10

    # Keep only sampled reaches
    df = df[df["reach_id"].isin(sampled_ids)]

    if not df.empty:
        train_valid_dfs.append(df)

    print(f"Selected train_valid data from {name}")

df_train_valid = pd.concat(train_valid_dfs, ignore_index=True)
train_valid_file = OUTPUT_DIR / "node_train_valid_v5.10.csv"
df_train_valid.to_csv(train_valid_file, index=False)

print(f"Saved combined train_valid dataset to {train_valid_file}")

In [ ]:
# Save the remaining reaches as continent-specific test datasets
reach_list_train = df_train_valid["reach_id"].unique()

for name in name_list:
    input_file = INPUT_DIR / f"node_{name}_v5.10.csv"
    output_file = OUTPUT_DIR / f"node_{name}_test_v5.10.csv"

    if not input_file.exists():
        continue

    df = pd.read_csv(input_file)

    # Derive reach_id from node_id
    df["reach_id"] = df["node_id"] // 10000 * 10 + df["node_id"] % 10

    # Keep only reaches not included in train_valid
    df = df[~df["reach_id"].isin(reach_list_train)]

    df.to_csv(output_file, index=False)
    print(f"Saved test dataset for {name} to {output_file}")

# 5 Compute mean sigma0

This step computes the average radar backscatter coefficient (σ0, `rdr_sig0`)
for each SWOT river node using the filtered continent-level datasets generated
in the previous steps.

The mean σ0 value is calculated for each `node_id` across all available
observations. The resulting dataset provides a node-level estimate of radar
backscatter that can be used for further analysis of river surface conditions.

**Input**  
Filtered continent-level node datasets:  
data/derived_raw/continent_valid_dates/  
&nbsp;&nbsp;&nbsp;&nbsp;node_af_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_eu_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_si_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_as_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_au_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_sa_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_na_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_ar_v5.10.csv  
&nbsp;&nbsp;&nbsp;&nbsp;node_gr_v5.10.csv  

**Output**  
Node-level mean σ0 dataset:  
data/input_data/node_sigma0_v5.10.csv  

**Notes**  
The σ0 values represent the mean radar backscatter coefficient observed at each
node across the filtered observation period. These statistics are used for
characterizing river surface scattering properties in subsequent analysis.

In [ ]:
# Continent short names
name_list = ["af", "eu", "si", "as", "au", "sa", "na", "ar", "gr"]

# Input/output directories
INPUT_DIR = PROJECT_ROOT / "data" / "derived_raw" / "continent_valid_dates"
OUTPUT_DIR = PROJECT_ROOT / "data" / "input_data"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

data_list = []

for name in name_list:
    input_file = INPUT_DIR / f"node_{name}_v5.10.csv"

    if not input_file.exists():
        print(f"File not found: {input_file}")
        continue

    df = pd.read_csv(input_file)
    print(f"Processing {name}...")

    # Compute mean sigma0 (rdr_sig0) for each node
    df_mean = (
        df.groupby("node_id", as_index=False)
        .agg({"rdr_sig0": "mean"})
    )

    data_list.append(df_mean)

# Merge all continent results
df_sigma0 = pd.concat(data_list, ignore_index=True)

# Save node-level sigma0 dataset
output_file = OUTPUT_DIR / "node_sigma0_v5.10.csv"
df_sigma0.to_csv(output_file, index=False)

print(f"Saved node-level sigma0 dataset to {output_file}")

# 6 Node geometry

This step extracts node-level geometry information from the raw daily node
datasets and builds a lookup table between `node_id` and node geometry.

Because node geometry should be fixed for a given `node_id`, this step first
collects all available `node_id`-`geometry` pairs from the raw data and then
resolves repeated records. If multiple geometry records exist for the same
node, the most frequently occurring geometry is retained.

**Input**  
Raw daily node datasets with geometry information:  
data/derived_raw/RiverSP_VC_daily/  
&nbsp;&nbsp;&nbsp;&nbsp;node_YYYYMMDD.csv  

**Output**  
Node geometry lookup table:  
data/derived_raw/node_geometry/  
&nbsp;&nbsp;&nbsp;&nbsp;node_geometry.csv  

**Notes**  
The datasets generated in this step are intermediate preprocessing results.
Due to their large size, they are not distributed with the repository but can
be regenerated using the provided scripts.

In [ ]:
# Input/output directories
INPUT_DIR = PROJECT_ROOT / "data" / "derived_raw" / "RiverSP_VC_daily"
OUTPUT_DIR = PROJECT_ROOT / "data" / "derived_raw" / "node_geometry"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Find all daily node files
node_files = sorted(glob(str(INPUT_DIR / "node_*.csv")))

if not node_files:
    raise ValueError(f"No node files found in {INPUT_DIR}")

geometry_records = []

for i, file in enumerate(node_files):
    df = pd.read_csv(file, usecols=["node_id", "geometry"])

    # Remove missing geometry records
    df = df.dropna(subset=["geometry"])

    geometry_records.append(df)

    if i % 5 == 0:
        print(f"Processed {i} files")

In [ ]:
# Merge all node-geometry records
df_geometry = pd.concat(geometry_records, ignore_index=True)

# Count how many distinct geometry strings exist for each node
geometry_counts = (
    df_geometry.groupby(["node_id", "geometry"])
    .size()
    .reset_index(name="count")
)

# Retain the most frequent geometry for each node_id
geometry_best = (
    geometry_counts.sort_values(["node_id", "count"], ascending=[True, False])
    .drop_duplicates(subset=["node_id"], keep="first")
    .drop(columns="count")
)

# Convert WKT strings to shapely geometry objects
geometry_best["geometry"] = geometry_best["geometry"].apply(wkt.loads)

# Build GeoDataFrame
gdf = gpd.GeoDataFrame(geometry_best, geometry="geometry")
gdf["lon"] = gdf.geometry.x
gdf["lat"] = gdf.geometry.y
df = pd.DataFrame(gdf[["node_id", "lon", "lat"]])

# Save output
output_file = OUTPUT_DIR / "node_geometry.csv"
df.to_csv(output_file, index=False)

print(f"Saved node geometry table to {output_file}")

# 7 GSW Water Occurrence

This step computes the mean Global Surface Water (GSW) occurrence value for each river node by sampling a 5×5 pixel window around the nearest pixel from the GSW occurrence raster tiles. The resulting values are used as static attributes for subsequent analyses.

**Input**  
Node geometry dataset:  
data/derived_raw/node_geometry/  
&nbsp;&nbsp;&nbsp;&nbsp;node_geometry.csv  

Global Surface Water occurrence tiles (downloaded from Internet):  
data/derived_raw/gsw_tiles/  
&nbsp;&nbsp;&nbsp;&nbsp;occurrence-*.tif  

**Output**  
Node-level mean occurrence values:  
data/input_data/  
&nbsp;&nbsp;&nbsp;&nbsp;node_occu_mean.csv

**Method**  
For each node coordinate (lon, lat), the script identifies the nearest raster pixel in the corresponding GSW occurrence tile. A 5×5 pixel window centered on this pixel is extracted, and the mean occurrence value within this window is calculated. The results are aggregated across all raster tiles and stored as node-level attributes.

**Notes**  
The Global Surface Water dataset can be obtained from:
https://global-surface-water.appspot.com/

Raster tiles must be downloaded and placed in data/derived_raw/gsw_tiles/ before running the script.

The output dataset provides a static water occurrence metric for each node, which is used in later modeling and analysis steps.

In [ ]:
# Input/output directories
INPUT_DIR = PROJECT_ROOT / "data" / "derived_raw" / "node_geometry"
RASTER_DIR = PROJECT_ROOT / "data" / "derived_raw" / "gsw_tiles"   # GSW occurrence raster tiles
OUTPUT_DIR = PROJECT_ROOT / "data" / "input_data"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def calculate_mean(row: pd.Series, ds) -> float:
    """
    Calculate the mean value of a 5x5 pixel window centered on the
    nearest raster pixel to a given coordinate.

    Parameters
    ----------
    row : pd.Series
        Row containing 'lon' and 'lat'.
    ds : xarray.DataArray or xarray.Dataset
        Raster opened with rioxarray.

    Returns
    -------
    float
        Mean value of the surrounding 5x5 pixel window.
    """

    # Select the nearest raster pixel
    nearest_pixel = ds.sel(x=row["lon"], y=row["lat"], method="nearest")

    # Get raster indices
    x_idx = np.abs(ds.x - nearest_pixel.x).argmin().item()
    y_idx = np.abs(ds.y - nearest_pixel.y).argmin().item()

    # Define a 5x5 window
    x_start = max(x_idx - 2, 0)
    x_end = min(x_idx + 3, ds.sizes["x"])
    y_start = max(y_idx - 2, 0)
    y_end = min(y_idx + 3, ds.sizes["y"])

    # Extract the local window
    area = ds.isel(x=slice(x_start, x_end), y=slice(y_start, y_end))

    # Return the mean value
    return float(area.mean().values)
def process_file(file_path: Path, df: pd.DataFrame) -> pd.DataFrame:
    """
    Process a single GSW raster tile and calculate occurrence values
    for nodes located within the raster bounds.

    Parameters
    ----------
    file_path : Path
        Path to a raster file.
    df : pd.DataFrame
        DataFrame containing 'node_id', 'lon', and 'lat'.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns ['node_id', 'occurrence'].
    """

    ds = rioxarray.open_rasterio(file_path).squeeze()

    # Keep only nodes within raster bounds
    in_bounds = (
        (df["lon"] > ds.x.min().values)
        & (df["lon"] < ds.x.max().values)
        & (df["lat"] > ds.y.min().values)
        & (df["lat"] < ds.y.max().values)
    )

    df_sub = df[in_bounds].copy()

    if df_sub.empty:
        return pd.DataFrame(columns=["node_id", "occurrence"])

    # Calculate mean occurrence values
    df_sub["occurrence"] = df_sub.apply(calculate_mean, axis=1, ds=ds)

    return df_sub[["node_id", "occurrence"]]
def parallel_processing(file_paths, df):
    """
    Run raster processing in parallel using dask.

    Parameters
    ----------
    file_paths : list
        List of raster file paths.
    df : pd.DataFrame
        Node coordinate DataFrame.

    Returns
    -------
    tuple[pd.DataFrame]
        Processed DataFrames for all raster tiles.
    """

    tasks = [
        dask.delayed(process_file)(file_path, df)
        for file_path in file_paths
    ]

    return dask.compute(*tasks)


In [ ]:
# Read node geometry table
node_file = INPUT_DIR / "node_geometry.csv"
df = pd.read_csv(node_file)

# Find GSW occurrence raster tiles
file_paths = glob(str(RASTER_DIR / "occurrence-*.tif"))

if not file_paths:
    raise ValueError(f"No raster files found in {RASTER_DIR}")

# Check required columns
required_columns = {"node_id", "lon", "lat"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(
        f"Missing required columns in node geometry table: {missing_columns}"
    )

# Run parallel raster processing
dfs = parallel_processing(file_paths, df)

# Merge results from all raster tiles
result_df = pd.concat(dfs, ignore_index=True)

# Save output
output_file = OUTPUT_DIR / "node_occu_mean.csv"
result_df.to_csv(output_file, index=False)

print(f"Saved node-level GSW occurrence dataset to {output_file}")